# Week 4 — Grouping, stratification, confounding

### Deliverable
- 2 subgroup tables + 2 plots + confounding note


In [4]:
# Core imports (kept minimal for beginners)
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 120)

# Dataset URL (UCI Heart Disease - Cleveland)
UCI_URL = "https://www.archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data"

# Column names for processed.cleveland.data (14 columns commonly used in teaching)
COLS = ["age","sex","cp","trestbps","chol","fbs","restecg","thalach",
        "exang","oldpeak","slope","ca","thal","num"]


In [1]:
import ssl
import io
import urllib.request # Added this import
def load_raw():
    # Create an unverified SSL context to bypass certificate verification
    ctx = ssl.create_default_context()
    ctx.check_hostname = False
    ctx.verify_mode = ssl.CERT_NONE

    # Open the URL with the unverified context
    with urllib.request.urlopen(UCI_URL, context=ctx) as url_response:
        # Read the content and decode it
        s = url_response.read().decode('utf-8')

    # Use io.StringIO to make the string behave like a file for pandas.read_csv
    df_raw = pd.read_csv(io.StringIO(s), header=None, names=COLS)
    return df_raw

def coerce_types(df_raw):
    # Missing values are sometimes encoded as "?"
    df = df_raw.replace("?", np.nan).copy()

    # Convert numeric-looking columns
    numeric_cols = ["age","trestbps","chol","thalach","oldpeak","ca","thal","num"]
    for c in numeric_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    # Binary target: disease present if num > 0 (UCI convention)
    df["disease"] = (df["num"] > 0).astype("Int64")
    return df

df_raw = load_raw()
df = coerce_types(df_raw)

df.head()


NameError: name 'UCI_URL' is not defined

In [ ]:
# Minimal cleaning for groupby (you can paste your Week 3 function instead)
def clean_heart(df_raw):
    df = df_raw.replace("?", np.nan).copy()
    numeric_cols = ["age","trestbps","chol","thalach","oldpeak","ca","thal","num"]
    for c in numeric_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    df["disease"] = (df["num"] > 0).astype("Int64")
    return df.dropna(subset=["disease","age","sex","cp"])

df_clean = clean_heart(df_raw)
df_clean.head()


## Step 1 — Create age bins


In [ ]:
bins = [0, 49, 59, 69, 120]
labels = ["<50", "50-59", "60-69", "70+"]
df_clean["age_bin"] = pd.cut(df_clean["age"], bins=bins, labels=labels, right=True)
df_clean["age_bin"].value_counts(dropna=False)


## Step 2 — Disease rates by subgroup


In [ ]:
overall = df_clean["disease"].mean()
print("Overall disease rate:", overall)

rate_by_sex = df_clean.groupby("sex")["disease"].agg(["mean","count"])
rate_by_age = df_clean.groupby("age_bin")["disease"].agg(["mean","count"])
rate_by_cp  = df_clean.groupby("cp")["disease"].agg(["mean","count"])

display(rate_by_sex)
display(rate_by_age)
display(rate_by_cp)


## Step 3 — Plots (include counts somewhere)


In [3]:
plt.figure()
rate_by_age["mean"].plot(kind="bar")
plt.ylabel("disease rate")
plt.title("Disease rate by age bin")
plt.show()

plt.figure()
rate_by_sex["mean"].plot(kind="bar")
plt.ylabel("disease rate")
plt.title("Disease rate by sex (coded 0/1)")
plt.show()


NameError: name 'plt' is not defined

## TODO — Confounding note
- Relationship you think exists:
- Possible confounder:
- How you will probe it (stratify / regression):


Confounding Note

Relationship you think exists:
There appears to be a strong association between gender and heart disease. In the table, the disease rate is approximately 55% in men (sex=1) and approximately 26% in women (sex=0). This difference suggests that men have more heart disease than women.

Possible confounder:
Age could be a potential confounder in this relationship because:

• The risk of heart disease increases with age (according to the age groups in the table, the rate is approximately 60% in the 60-69 age group, 48% in 50-59, and 30% in those under 50).
• The average age in the population may differ between genders (for example, men in the study population may be older).
If the average age of men is higher than that of women, some of the observed gender difference may actually be due to age. In this case, age plays a confounding role in the relationship between gender and disease.

How to Probe It (Stratify/Regression):

1. Stratification:
I calculate disease rates separately for each age group (<50, 50-59, 60-69, 70+) based on gender. If the gender difference decreases or disappears within each age group, it proves that age is confounding.

2. Regression (Logistic Regression):
I build a logistic regression model that includes both age and gender. The model includes disease ~ sex + age (continuous age or categorical). If the gender coefficient loses significance or decreases significantly after controlling for age, this supports the idea that age is confounding.
